In [ ]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("../config/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 11))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


In [ ]:
df = df_orig.sample(500)

In [3]:
from scripts.beamforming import get_best_beam

df["best_beam"] = df["measurements_matrix"].apply(
    lambda x: get_best_beam(x, rf_param)
)

In [4]:
df

,lat,lng,measurements_matrix,campaign_id,best_beam
3625,41.898485,12.428332,pci beam_index nr_arfcn operator_id ...,8,"(-108.0, 5.0, 643296.0, 10.0)"
4372,41.896414,12.428538,pci beam_index nr_arfcn operator_id ...,10,"(-108.0, 2.0, 643296.0, 10.0)"
4415,41.894920,12.429491,pci beam_index nr_arfcn operator_id ...,10,"(-108.0, 1.0, 643296.0, 10.0)"
2073,41.898489,12.428800,pci beam_index nr_arfcn operator_id ...,5,"(75.0, 4.0, 643296.0, 10.0)"
3979,41.894838,12.429736,pci beam_index nr_arfcn operator_id ...,9,"(76.0, 6.0, 643296.0, 10.0)"
...,...,...,...,...,...
3663,41.898472,12.429247,pci beam_index nr_arfcn operator_id ...,8,"(-109.0, 1.0, 643296.0, 10.0)"
361,41.898483,12.428550,pci beam_index nr_arfcn operator_id ...,1,"(-108.0, 5.0, 643296.0, 10.0)"
295,41.898227,12.427951,pci beam_index nr_arfcn operator_id ...,1,"(-109.0, 1.0, 643296.0, 10.0)"
2990,41.897076,12.428533,pci beam_index nr_arfcn operator_id ...,7,"(76.0, 7.0, 643296.0, 10.0)"


In [5]:
from scripts.utils import dataset_tp_rp_split, extract_unique_npcis
from scripts.weighted_coverage import create_point_matrix, compute_weights
from scripts.beamforming import filter_best_beam
import pandas as pd

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 420)
df_rp_ctrl = df_rp.copy()
df_rp_filter = df_rp.copy()

all_pcis = extract_unique_npcis(df['measurements_matrix'])

tp = df_tp.iloc[7]

best_beam = tp["best_beam"]
best_beam

tp = pd.DataFrame([tp])

# Calculate the point matricies with rf values for all RPs and TP using only the best beam PCI
m_rp, idx_rp = create_point_matrix(df_rp, [best_beam], rf_param)
m_tp, idx_tp = create_point_matrix(tp, [best_beam], rf_param)
m_tp_ctrl, idx_tp_ctrl = create_point_matrix(tp, all_pcis, rf_param)
W, idx_sort = compute_weights(m_rp, idx_rp, m_tp, idx_tp)

# Calculate the point matricies with rf values for all RPs and TP using all PCIs

m_rp_ctrl, idx_rp_ctrl = create_point_matrix(df_rp_ctrl, all_pcis, rf_param)
m_tp_ctrl, idx_tp_ctrl = create_point_matrix(tp, all_pcis, rf_param)
W_ctrl, idx_sort_ctrl = compute_weights(m_rp_ctrl, idx_rp_ctrl, m_tp_ctrl, idx_tp_ctrl)

# Calculate the point matricies with rf values for all RPs and TP using all PCIs, with filtered measurements matrix
tp_filter = tp.copy()

# fitler the dataframes measurements matrix
tp_filter.loc[:, "measurements_matrix"] = tp_filter.loc[:, "measurements_matrix"].apply(
    lambda x: filter_best_beam(x, rf_param)
)
df_rp_filter.loc[:, "measurements_matrix"] = df_rp_filter.loc[:, "measurements_matrix"].apply(
    lambda x: filter_best_beam(x, rf_param)
)
m_rp_filter, idx_rp_filter = create_point_matrix(df_rp_filter, all_pcis, rf_param)
m_tp_filter, idx_tp_filter = create_point_matrix(tp_filter, all_pcis, rf_param)
W_filter, idx_sort_filter = compute_weights(m_rp_filter, idx_rp_filter, m_tp_filter, idx_tp_filter)

print(f"""
Control (without using only best beam)
Selected RPs indecies \t {idx_sort_ctrl[0, :10]}

Test (using only best beam)
Selected RPs indecies \t {idx_sort[0, :10]}

Test (using best beam filtering)
Selected RPs indecies \t {idx_sort_filter[0, :10]}
""")


Control (without using only best beam)
Selected RPs indecies 	 [282 166 334 174 260  38 235 123 172 321]

Test (using only best beam)
Selected RPs indecies 	 [166 282  30 174 123 235 172 321  38 120]

Test (using best beam filtering)
Selected RPs indecies 	 [166 282  30 174 235  38 125  56 115 230]



In [7]:
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from scripts.weighted_coverage import wknn


def compute_weights_pca(m_rfp_pca, m_tp_pca):
    """
    Compute weights using PCA-transformed data
    """
    # Compute Euclidean distances in the PCA space
    D = cdist(m_tp_pca, m_rfp_pca, metric="euclidean")

    # Sort distances and compute weights
    idx_sort = np.argsort(D, axis=1)
    D_sort = np.take_along_axis(D, idx_sort, axis=1)

    # Avoid division by zero
    min_nonzero_distance = np.min(D[D > 0]) if np.any(D > 0) else 0.1
    D_sort[D_sort == 0] = min_nonzero_distance / 20

    W = 1.0 / D_sort
    return W, idx_sort


def pca_strategy(df_tp: pd.DataFrame, df_rp: pd.DataFrame, pcis: list[tuple], rf_param: RF_PARAM_5G):
    m_rp_full, idx_rp_full = create_point_matrix(df_rp, pcis, rf_param)
    m_tp_full, idx_tp_full = create_point_matrix(tp, pcis, rf_param)

    pca = PCA(n_components=0.95)
    pca.fit(m_rp_full)
    m_rp_pca = pca.transform(m_rp_full)
    m_tp_pca = pca.transform(m_tp_full)

    W_pca, idx_sort_pca = compute_weights_pca(m_rp_pca, m_tp_pca)

    _, errors = wknn(df_tp, df_rp, W_pca, idx_sort_pca, k=2)

    return errors


def normal_strategy(df_tp: pd.DataFrame, df_rp: pd.DataFrame, pcis: list[tuple], rf_param: RF_PARAM_5G):
    m_rp_full, idx_rp_full = create_point_matrix(df_rp, pcis, rf_param)
    m_tp_full, idx_tp_full = create_point_matrix(tp, pcis, rf_param)

    W, idx_sort = compute_weights(m_rp_full, idx_rp_full, m_tp_full, idx_tp_full)

    _, errors = wknn(df_tp, df_rp, W, idx_sort, k=2)

    return errors


res_pca = pca_strategy(tp, df_rp, all_pcis, rf_param)
res_normal = normal_strategy(tp, df_rp, all_pcis, rf_param)

print(f"""
PCA {res_pca}
Normal {res_normal}
""")


PCA [125.08525777]
Normal [452.21232557]

